# Densest subgraph

## Basic Graph Class

We already provide the basic implementation of a graph class.

Iterating over nodes can be done by using ```for u in G.nodesToEdges``` and iterating over the edges incident upon node ```u``` can be done using ```for v in G.nodesToEdges[u]```. To check whether an edge $(u,v)$ exists use ```G.edgeExists(u,v)```.

Graphs can be read from files using the ```readFromFile```-procedure. Using the optional argument ```verticesToIgnore``` which expects a ```set()``` as input, you may exclude some nodes from the graph, i.e., if ```verticesToIgnore``` is some set $S$ then it will load the graph $G=(V\setminus S,E[V\setminus S])$.  Note that you might have to delete header-rows from some data files to make the procedure work.

In [67]:
class Graph:
    def __init__(self):
        self.numNodes = 0
        self.numEdges = 0

        self.edges = set()
        self.nodesToEdges = {}

    def addEdge(self,u,v):
        if u == v or v in self.nodesToEdges and u in self.nodesToEdges[v]:
            return
            
        self.numEdges += 1
        self.addNeighbor(u,v)
        self.addNeighbor(v,u)
        self.edges.add((u,v)) # we only add one pair (u,v) and not (v,u)

    def removeEdge(self,u,v):
        self.numEdges -= 1
        self.nodesToEdges[u].remove(v)
        self.nodesToEdges[v].remove(u)

        if (u,v) in self.edges:
            self.edges.remove((u,v))
        else: # (v,u) in self.edges
            self.edges.remove((v,u))

    def addNeighbor(self,u,v):
        if u not in self.nodesToEdges:
            self.numNodes += 1
            self.nodesToEdges[u] = set()

        self.nodesToEdges[u].add(v)

    def edgeExists(self, u, v):
        return ((u,v) in self.edges or (v,u) in self.edges)
        
    def degree(self, u):
        return len(self.nodesToEdges[u])

    def readFromFile(self, filePath, separator=',', verticesToIgnore=set()):
        with open(filePath, 'r') as f:
            for line in f:
                split = line.split(separator)
                u = int(split[0])
                v = int(split[1])

                if u in verticesToIgnore or v in verticesToIgnore:
                    continue
                    
                self.addEdge(u, v)

        print(f'Finished reading graph with {self.numNodes} nodes and {self.numEdges} edges.')

## Basic Linked List Data Structure

First, implement your own linked list data structure ```LinkedList``` in which each element is from the class ```LinkedListElement```.

In [68]:
class LinkedListElement:
    def __init__(self,id,prevElement=None,nextElement=None):
        self.id = id
        self.prevElement = prevElement
        self.nextElement = nextElement

class LinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def appendElement(self, element: LinkedListElement):
        if self.head is None:
            self.head = element
            self.tail = element
        else:
            element.prevElement = self.tail
            self.tail.nextElement = element
            self.tail = element
        self.size += 1

    def removeElement(self, element: LinkedListElement):
        if element == self.head:
            self.head = element.nextElement

        if element == self.tail:
            self.tail = element.prevElement

        if element.prevElement:
            element.prevElement.nextElement = element.nextElement
        if element.nextElement:
            element.nextElement.prevElement = element.prevElement

        element.nextElement = None
        element.prevElement = None
        self.size -= 1
        
    def pop(self):
        if not self.tail:
            return None
        node_to_remove = self.tail
        self.removeElement(node_to_remove)
        return node_to_remove

## Implementation of the Greedy Peeling Algorithm

Next, implement the greedy peeling algorithm. The function should be callable using ```densestSubgraphGreedyPeeling(G)``` where ```G``` is a graph using the graph class from above. Besides G, the function may accept more optional arguments.

In [69]:
def objectiveValue(num_edges, num_nodes, objective='density'):
    match objective:
        case 'density':
            return num_edges / num_nodes
        case 'tsourakakis ':
            return num_edges - (num_nodes * (num_nodes - 1)) / 6
        case _:
            raise ValueError(f'Unknown objective: {objective}')

def densestSubgraphGreedyPeeling(G, objective='density'):
    max_id = max(G.nodesToEdges) # needed to allow non-continuous labeling
    d = [0] * (max_id + 1)
    # im assuming here that the vertices are named 0 - n <-- now also fine with a few "holes"
    for e in G.edges:
        d[e[0]] += 1
        d[e[1]] += 1

    L = list()
    for _ in range(0, G.numNodes):
        L.append(LinkedList())

    k_star = float('inf')
    pointers_to_vertices = [None] * (max_id + 1)  # needed to allow non-continuous labeling
    #for v, degree in enumerate(d):
    for v in G.nodesToEdges:   # works with a non-continuous degree list
        degree = d[v]
        element = LinkedListElement(v)
        L[degree].appendElement(element)
        pointers_to_vertices[v] = element

        if degree < k_star:
            k_star = degree

    #best_S = G.edges.copy() # O(m)
    best_num_edges = G.numEdges
    best_score = objectiveValue(G.numEdges, G.numNodes, objective)
    removed_vertices = []
    best_removed_count = 0
    num_nodes = G.numNodes
    while num_nodes != 1:
        v_star = L[k_star].pop()
        removed_vertices.append(v_star.id)
        neighbors = G.nodesToEdges[v_star.id].copy() # O(m)
        for u in neighbors:
            G.removeEdge(v_star.id, u)
            current = pointers_to_vertices[u]
            L[d[current.id]].removeElement(current) # remove from L[d[u]]
            d[current.id] -= 1
            L[d[current.id]].appendElement(current) # add to L[d[u]-1]

            if d[current.id] < k_star:
                k_star = d[current.id]
        num_nodes -= 1


        score = objectiveValue(G.numEdges, num_nodes, objective)
        if score > best_score:
            best_num_edges = G.numEdges
            best_score = score
            #best_S = G.edges.copy() # O(m) <- we are inside a O(n) loop, this would result in O(n*m)
            best_removed_count = len(removed_vertices)

        while k_star < len(L) and L[k_star].size <= 0:
            k_star = (k_star + 1) % G.numNodes

    best_S = set(removed_vertices[best_removed_count:]) # "get back" all vertices that were removed, after optimum was found
    best_S.add(L[k_star].head.id) # also add last vertex still in the list

    return best_S, best_num_edges, best_score

def runDisjointSubgraphExtraction(filePath, objective='density'):
    verticesToIgnore = set()
    results = []

    for round_idx in range(1, 6):
        working_graph = Graph()
        working_graph.readFromFile(filePath, ',', verticesToIgnore)
        vertex_set, _, density = densestSubgraphGreedyPeeling(working_graph)

        results.append({
            'round': round_idx,
            'vertex_set': set(vertex_set),
            'num_vertices': len(vertex_set),
            'density': density,
        })
        verticesToIgnore.update(vertex_set)

    return results

testGraph = Graph()
testGraph.addEdge(0,1)
testGraph.addEdge(1,2)
testGraph.addEdge(1,3)
testGraph.addEdge(1,25)
testGraph.addEdge(2,3)
testGraph.addEdge(2,25)
testGraph.addEdge(3,25)
densestSubgraphGreedyPeeling(testGraph)

({1, 2, 3, 25}, 6, 1.5)

## Experiments for Densest Subgraph

Now, the experiments for the two datasets follow.

### Experiments for the OpenFlights Dataset

In [70]:
openflights_disjoint_subgraphs = runDisjointSubgraphExtraction('data/openflights_edges_cleaned.csv')

for result in openflights_disjoint_subgraphs:
    print(f"Round {result['round']}:")
    print(f"|S| = {result['num_vertices']}, density = {result['density']}\n")


Finished reading graph with 3214 nodes and 18858 edges.
Finished reading graph with 2643 nodes and 7349 edges.
Finished reading graph with 2513 nodes and 5716 edges.
Finished reading graph with 2415 nodes and 4953 edges.
Finished reading graph with 2234 nodes and 4027 edges.
Round 1:
|S| = 180, density = 25.377777777777776

Round 2:
|S| = 53, density = 14.754716981132075

Round 3:
|S| = 29, density = 7.482758620689655

Round 4:
|S| = 57, density = 5.385964912280702

Round 5:
|S| = 18, density = 4.611111111111111



### AI-Assisted OpenFlights Visualizations

The following cell was written with AI assistance. It creates one overview world map for all five OpenFlights clusters and five additional cluster-specific maps that also display the intra-cluster flight edges. The figures are shown in the notebook and saved as PNG files in the `generated/` directory.

In [ ]:
# AI-assisted visualization code.
# It creates one overview map for all clusters and one route map for each cluster.

import csv
import os
import plotly.graph_objects as go

with open('data/openflights/nodes.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    openflights_nodes = {
        int(row[0]): {
            'name': row[2],
            'city': row[3],
            'country': row[4],
            'lat': float(row[7]),
            'lon': float(row[8]),
        }
        for row in reader
    }

openflights_edges = set()
with open('data/openflights_edges_cleaned.csv', newline='', encoding='utf-8') as f:
    for row in csv.reader(f):
        u = int(row[0])
        v = int(row[1])
        if u == v:
            continue
        openflights_edges.add((u, v) if u < v else (v, u))

cluster_colors = ['#d1495b', '#edae49', '#66a182', '#2e86ab', '#5c4d7d']

def airport_hover_text(round_idx, node):
    return f"Round {round_idx}<br>{node['name']} ({node['city']}, {node['country']})"

def cluster_nodes(vertex_set):
    node_ids = sorted(vertex_set)
    return node_ids, [openflights_nodes[node_id] for node_id in node_ids]

def cluster_edge_coordinates(vertex_set):
    vertex_set = set(vertex_set)
    lon = []
    lat = []
    edge_count = 0
    for u, v in openflights_edges:
        if u in vertex_set and v in vertex_set:
            lon.extend([openflights_nodes[u]['lon'], openflights_nodes[v]['lon'], None])
            lat.extend([openflights_nodes[u]['lat'], openflights_nodes[v]['lat'], None])
            edge_count += 1
    return lon, lat, edge_count

def padded_range(values, lower_bound, upper_bound):
    minimum = min(values)
    maximum = max(values)
    span = maximum - minimum
    if span > 300:
        return None
    padding = max(5, span * 0.15)
    return [max(lower_bound, minimum - padding), min(upper_bound, maximum + padding)]

def apply_geo_style(fig, title, lon_range=None, lat_range=None):
    geo_options = dict(
        projection_type='natural earth',
        showland=True,
        landcolor='#efefef',
        showcountries=True,
        countrycolor='#ffffff',
        showocean=True,
        oceancolor='#dcefff',
        showcoastlines=True,
        coastlinecolor='#888888',
    )
    if lon_range is not None:
        geo_options['lonaxis'] = dict(range=lon_range)
    if lat_range is not None:
        geo_options['lataxis'] = dict(range=lat_range)
    fig.update_geos(**geo_options)
    fig.update_layout(
        title=title,
        legend_title_text='Cluster',
        width=1200,
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )

os.makedirs('generated', exist_ok=True)

overview_fig = go.Figure()
for result, color in zip(openflights_disjoint_subgraphs, cluster_colors):
    _, nodes = cluster_nodes(result['vertex_set'])
    overview_fig.add_trace(go.Scattergeo(
        lon=[node['lon'] for node in nodes],
        lat=[node['lat'] for node in nodes],
        text=[airport_hover_text(result['round'], node) for node in nodes],
        mode='markers',
        marker=dict(size=7, color=color, opacity=0.85, line=dict(width=0.5, color='white')),
        name=f"Round {result['round']} (|S|={result['num_vertices']})",
        hovertemplate="%{text}<extra></extra>",
    ))

apply_geo_style(overview_fig, 'OpenFlights Dense Subgraphs on a World Map')
overview_fig.show()
overview_path = os.path.join('generated', 'openflights_clusters_world_map.png')
overview_fig.write_image(overview_path, width=1200, height=700, scale=2)
print(f"Saved overview map to {overview_path}")

for result, color in zip(openflights_disjoint_subgraphs, cluster_colors):
    node_ids, nodes = cluster_nodes(result['vertex_set'])
    edge_lon, edge_lat, edge_count = cluster_edge_coordinates(node_ids)

    cluster_fig = go.Figure()
    cluster_fig.add_trace(go.Scattergeo(
        lon=edge_lon,
        lat=edge_lat,
        mode='lines',
        line=dict(width=1, color=color),
        opacity=0.18,
        hoverinfo='skip',
        showlegend=False,
    ))
    cluster_fig.add_trace(go.Scattergeo(
        lon=[node['lon'] for node in nodes],
        lat=[node['lat'] for node in nodes],
        text=[airport_hover_text(result['round'], node) for node in nodes],
        mode='markers',
        marker=dict(size=8, color=color, opacity=0.9, line=dict(width=0.6, color='white')),
        name=f"Round {result['round']} airports",
        hovertemplate="%{text}<extra></extra>",
    ))

    lon_range = padded_range([node['lon'] for node in nodes], -180, 180)
    lat_range = padded_range([node['lat'] for node in nodes], -90, 90)
    apply_geo_style(
        cluster_fig,
        f"OpenFlights Cluster {result['round']} with Intra-Cluster Routes (|S|={result['num_vertices']}, |E[S]|={edge_count})",
        lon_range,
        lat_range,
    )
    cluster_fig.show()

    cluster_path = os.path.join('generated', f"openflights_cluster_{result['round']}.png")
    cluster_fig.write_image(cluster_path, width=1200, height=700, scale=2)
    print(f"Saved cluster map to {cluster_path}")


### Experiments for the Facebook Dataset

In [ ]:
import csv

facebook_disjoint_subgraphs = runDisjointSubgraphExtraction('data/musae_facebook_edges_cleaned.csv')

for result in facebook_disjoint_subgraphs:
    print(f"Round {result['round']}:")
    print(f"|S| = {result['num_vertices']}, density = {result['density']}\n")

with open('data/facebook_large/musae_facebook_target.csv', newline='', encoding='utf-8') as f:
    facebook_page_names = {
        int(row['id']): row['page_name']
        for row in csv.DictReader(f)
    }

facebook_densest_page_names = sorted(
    facebook_page_names[node_id]
    for node_id in facebook_disjoint_subgraphs[0]['vertex_set']
)

facebook_densest_page_names

## Experiments for Optimal Quasi Clique

Now re-run the experiments from above with the new objective function and compare the results to the previous outputs.

### Experiments for the OpenFlights Dataset

### Experiments for the Facebook Dataset